# ABS Labour Force Survey data

The notebook’s goal is to automate the monthly preparation of ABS Labour Force data for analysis and dashboards. It identifies the next release to process, downloads three workbooks covering national and state labour force figures, youth figures, and modelled regional estimates, then checks them against reference files for structural or definition changes. It reshapes the data into consistent, clearly labelled tables and saves three cleaned CSVs, while retaining the original downloads and a source manifest so each completed run is traceable.

- **Table 010:** Labour force status by Sex, State and Territory — Trend, Seasonally adjusted and Original
- **Table 014:** Labour force status for 15–24 year olds by State and Territory
- **MLF1:** Modelled estimates of Labour force status by SA4 (ASGS), Age and Sex

**Outputs:** three cleaned CSV files named for their source table and reference month, plus a source manifest, in a new dated folder under `outputs/abs_labour_force/`. Downloads are retained under `raw/` in that folder, written only once all three have downloaded. The source manifest is written only after all three clean files are saved; earlier successful runs are kept separately.

### What is a material change?

| Stop the run | Allow without changing the baseline |
|---|---|
| Missing or additional sheets; moved header rows; changed MLF1 table identity | New monthly observations |
| Added, removed or duplicate series IDs; changed descriptions, units, frequency, series type, or historical start | Revisions to numeric estimates |
| Changed MLF1 columns, age/sex categories, or region codes/names (including geography vintage) | Reordered series columns within a data sheet; reordered MLF1 rows |
| Blank or invalid estimates in MLF1; MLF1 months with differing row counts | Cosmetic formatting; copyright and release-date updates |
| Latest month in any workbook is not the release being processed | |

## 1. Setup and reference workbooks

Use Python 3.10+ with `pandas`, `openpyxl`, `requests` and a Jupyter kernel. In a new environment, run `%pip install pandas openpyxl requests` once, then restart the kernel. Paths assume the notebook is opened from the project folder.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import requests
from IPython.display import display

root = Path.cwd()
baseline = root / 'Labour Force Datasets'
now = datetime.now(timezone.utc)
retrieved_at = now.isoformat()
run = root / 'outputs' / 'abs_labour_force' / now.strftime('%Y%m%dT%H%M%S%fZ')

## 2. Build the next release URLs and download all three files

The next release is one month after the later of the last month in the reference Table 010 workbook and the latest completed run. For the three workbooks, build the observed ABS URL pattern from that month and each fixed filename. The download fails if a file is unavailable. All three are held in memory and the run folder is created only after every download succeeds, so trying an unpublished release leaves nothing behind. MLF1 is scheduled to move to a separate ABS publication from October 2026, so its URL will need updating then.

In [2]:
sources = pd.DataFrame([
    ('Table 010', 'Table 010.xlsx', '62020010.xlsx'),
    ('Table 014', 'Interim Table 14.xlsx', '62020014.xlsx'),
    ('MLF1', 'MLF1.xlsx', 'MLF1.xlsx'),
], columns=['dataset', 'baseline_file', 'download_file']).set_index('dataset')

reference_file = baseline / sources.loc['Table 010', 'baseline_file']
reference_month = pd.to_datetime(pd.read_excel(reference_file, sheet_name='Data1', header=None).iloc[-1, 0])
completed = (root / 'outputs' / 'abs_labour_force').glob('*/sources.csv')
processed = [pd.to_datetime(pd.read_csv(p)['reference_period']).max() for p in completed]
reference_period = max([reference_month, *processed]) + pd.offsets.MonthBegin(1)

site = 'https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/'
sources['source_url'] = f"{site}{reference_period:%b-%Y}/".lower() + sources['download_file']
sources['reference_period'] = reference_period
sources['retrieved_at_utc'] = retrieved_at

downloads = {}
for row in sources.itertuples():
    r = requests.get(row.source_url, timeout=120, headers={'Cache-Control': 'no-cache'})
    r.raise_for_status()
    assert r.content.startswith(b'PK'), f'{row.Index}: download is not an XLSX file'
    downloads[row.download_file] = r.content

(run / 'raw').mkdir(parents=True)
for name, content in downloads.items():
    (run / 'raw' / name).write_bytes(content)

print('ABS reference month:', f'{reference_period:%B %Y}')
display(sources[['download_file', 'reference_period', 'source_url']])

ABS reference month: August 2026


,download_file,reference_period,source_url
dataset,,,
Table 010,62020010.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...
Table 014,62020014.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...
MLF1,MLF1.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...


## 3. Read the time-series workbooks

Tables 010 and 014 share an ABS time-series layout: ten metadata rows followed by monthly values. Read every populated data column. Expected sheets and header labels come from the current reference workbooks.

For each data sheet, the first seven metadata fields must match the reference by ABS series ID, and the latest month must be the release being processed. Observation counts, monthly continuity and numeric completeness are not separately checked. The monthly values are reshaped into one row per date and series for the clean table.

In [3]:
base_books, books = {}, {}
for dataset in ['Table 010', 'Table 014']:
    row = sources.loc[dataset]
    base_books[dataset] = pd.read_excel(baseline / row['baseline_file'], sheet_name=None, header=None)
    books[dataset] = pd.read_excel(run / 'raw' / row['download_file'], sheet_name=None, header=None)
    assert set(books[dataset]) == set(base_books[dataset]), f'{dataset}: workbook sheets changed'

labels = ['description', 'unit', 'series_type', 'data_type', 'frequency', 'collection_month',
          'series_start', 'series_end', 'n_obs', 'series_id']
metadata, observations, checks = [], [], []
for dataset, book in books.items():
    for sheet, raw in book.items():
        if not sheet.startswith('Data'):
            continue
        base = base_books[dataset][sheet]
        pd.testing.assert_series_equal(raw.iloc[:10, 0], base.iloc[:10, 0], obj=f'{dataset}/{sheet}: header rows')
        assert raw.iloc[9, 1:].notna().all() and raw.iloc[9, 1:].is_unique, f'{dataset}/{sheet}: missing or duplicate series IDs'

        # Matching by series ID lets reordered columns pass, while added, removed or redefined series stop the run.
        meta, expected = (
            d.iloc[:10, 1:].T.set_axis(labels, axis=1).set_index('series_id').sort_index()
            for d in (raw, base)
        )
        pd.testing.assert_frame_equal(meta.iloc[:, :7], expected.iloc[:, :7], obj=f'{dataset}/{sheet}: series definitions')

        long = (
            raw.iloc[10:]
               .set_axis(['date', *raw.iloc[9, 1:]], axis=1)
               .melt('date', var_name='series_id')
               .astype({'date': 'datetime64[ns]', 'value': float})
        )
        assert long['date'].max() == reference_period, f'{dataset}/{sheet}: latest month is not {reference_period:%B %Y}'
        observations.append(long.assign(dataset=dataset))
        metadata.append(meta.reset_index().assign(dataset=dataset, sheet=sheet))
        checks.append((dataset, sheet, long['date'].nunique(), len(meta), len(long), long['date'].max()))

meta = pd.concat(metadata, ignore_index=True)
ts = pd.concat(observations, ignore_index=True)
assert not meta.duplicated(['dataset', 'series_id']).any(), 'A series ID is repeated across sheets'
display(pd.DataFrame(checks, columns=['dataset', 'sheet', 'months', 'series', 'observations', 'latest_month']))

,dataset,sheet,months,series,observations,latest_month
0,Table 010,Data1,583,250,145750,2026-08-01
1,Table 010,Data2,583,250,145750,2026-08-01
2,Table 010,Data3,583,244,142252,2026-08-01
3,Table 014,Data1,485,90,43650,2026-08-01


### Split the descriptions into clean dimensions

Assumptions: Table 010 covers people aged 15 years and over; Table 014 covers persons aged 15–24. ABS hierarchy markers (`>`) are removed only after the original descriptions have passed validation. Published units are retained: `000` means thousands of people and `Percent` stays as a percentage: 4.5 means 4.5%, not 0.045.

In [4]:
parts = meta['description'].str.strip().str.rstrip(';').str.split(';', expand=True)
parts = parts.apply(lambda s: s.str.strip().str.lstrip('>').str.strip())
meta['measure'] = parts[0]
meta['sex'] = parts[1]
meta['region'] = parts[2]
meta['age'] = '15 years and over'

# Table 014 descriptions put the region first and are not split by sex.
youth = meta['dataset'].eq('Table 014')
meta.loc[youth, 'region'] = parts.loc[youth, 0]
meta.loc[youth, 'measure'] = parts.loc[youth, 1]
meta.loc[youth, 'sex'] = 'Persons'
meta.loc[youth, 'age'] = '15-24 years'
assert meta[['measure', 'sex', 'region', 'age']].notna().all().all()

ts = ts.merge(
    meta[['dataset', 'series_id', 'measure', 'sex', 'age', 'region', 'series_type', 'data_type', 'unit']],
    on=['dataset', 'series_id'], validate='many_to_one'
)
print(ts.shape)
display(ts[['date', 'series_id', 'measure', 'region', 'sex', 'age', 'series_type', 'data_type', 'unit', 'value']].head())

(477402, 11)


,date,series_id,measure,region,sex,age,series_type,data_type,unit,value
0,1978-02-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6007.747586
1,1978-03-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6014.722083
2,1978-04-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6021.174221
3,1978-05-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6026.885932
4,1978-06-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6030.961427


## 4. Read and validate MLF1 regional estimates

Use the underlying **Data 1** sheet, which contains all sex × age × region combinations. The presentation pivot sheet is not another dataset and is not appended. Ignore only completely empty columns below the workbook preamble; any populated new header or data column is compared with the baseline. Every month must have the same number of rows, and the latest month must be the release being processed.

In [5]:
row = sources.loc['MLF1']
base_xl = pd.ExcelFile(baseline / row['baseline_file'])
xl = pd.ExcelFile(run / 'raw' / row['download_file'])
assert set(xl.sheet_names) == set(base_xl.sheet_names), 'MLF1: workbook sheets changed'

mlf_base = pd.read_excel(base_xl, sheet_name='Data 1', header=None)
mlf_raw = pd.read_excel(xl, sheet_name='Data 1', header=None)
assert mlf_raw.iloc[1, 0] == mlf_base.iloc[1, 0], 'MLF1: table identity changed'

# Row 3 holds the column headings; everything above it is preamble.
mlf_base, mlf_raw = (d.iloc[3:].dropna(axis=1, how='all') for d in (mlf_base, mlf_raw))
assert mlf_raw.iloc[0].tolist() == mlf_base.iloc[0].tolist(), 'MLF1: column names, units, order or geography vintage changed'
mlf_base, mlf_raw = (
    d.iloc[1:].set_axis(d.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()
    for d in (mlf_base, mlf_raw)
)

dims = mlf_raw.columns[1:4].tolist()  # sex, age and SA4 region
measures = mlf_raw.columns[4:].tolist()
assert mlf_raw.notna().all().all(), 'MLF1: missing dimension or estimate'
assert mlf_raw[measures].dtypes.map(pd.api.types.is_numeric_dtype).all(), 'MLF1: non-numeric observations'
for col in dims:
    assert set(mlf_raw[col]) == set(mlf_base[col]), f'MLF1: categories changed in {col}'

mlf_raw['Month'] = pd.to_datetime(mlf_raw['Month'])
counts = mlf_raw.groupby('Month').size()
assert counts.nunique() == 1, 'MLF1: months have different numbers of sex × age × region rows'
assert mlf_raw['Month'].max() == reference_period, f'MLF1: latest month is not {reference_period:%B %Y}'

print(mlf_raw.shape)
display(mlf_raw.head())
display(pd.DataFrame({
    'months': [len(counts)], 'regions': [mlf_raw[dims[2]].nunique()],
    'sex_categories': [mlf_raw['Sex'].nunique()], 'age_groups': [mlf_raw['Age'].nunique()],
    'rows_per_month': [counts.iloc[0]], 'latest_month': [mlf_raw['Month'].max()],
}))

(183744, 7)


,Month,Sex,Age,Labour market region (SA4): ASGS (2011),Employed total ('000),Unemployed total ('000),Not in the labour force (NILF) ('000)
0,2012-01-01,Males,15-24 years,102 Central Coast,12.757092,2.090347,5.996909
1,2012-01-01,Males,15-24 years,115 Sydney - Baulkham Hills and Hawkesbury,9.284133,1.077604,5.995995
2,2012-01-01,Males,15-24 years,116 Sydney - Blacktown,12.403918,2.240202,8.865894
3,2012-01-01,Males,15-24 years,117 Sydney - City and Inner South,11.087451,1.585265,7.928945
4,2012-01-01,Males,15-24 years,118 Sydney - Eastern Suburbs,8.474928,1.107462,8.446889


,months,regions,sex_categories,age_groups,rows_per_month,latest_month
0,176,87,2,6,1044,2026-08-01


### Tidy regional measures

The workbook labels its geography **Labour market region (SA4): ASGS (2011)**. Preserve that definition and the published region names, including combined regions. `series_type = Modelled` identifies this source; it does not assert a seasonal adjustment status. MLF1 does not supply ABS time-series IDs, so no series ID is exported. Only the underlying Data 1 regional records are used; national and state totals from the presentation pivot are not imported. No totals, rates or additional populations are inferred from the pivot table.

In [6]:
mlf = (
    mlf_raw.rename(columns={'Month': 'date', 'Sex': 'sex', 'Age': 'age', dims[2]: 'region'})
           .melt(id_vars=['date', 'sex', 'age', 'region'], value_vars=measures, var_name='measure')
)
mlf[['region_code', 'region']] = mlf['region'].str.extract(r'^(\d{3}) (.+)$')
assert mlf['region_code'].notna().all(), 'MLF1: region-code format changed'
mlf['measure'] = mlf['measure'].str.replace(" ('000)", '', regex=False)
mlf['unit'] = '000'
mlf['series_type'] = 'Modelled'
print(mlf.shape)
display(mlf[['date', 'region_code', 'region', 'sex', 'age', 'measure', 'value']].head())

(551232, 9)


,date,region_code,region,sex,age,measure,value
0,2012-01-01,102,Central Coast,Males,15-24 years,Employed total,12.757092
1,2012-01-01,115,Sydney - Baulkham Hills and Hawkesbury,Males,15-24 years,Employed total,9.284133
2,2012-01-01,116,Sydney - Blacktown,Males,15-24 years,Employed total,12.403918
3,2012-01-01,117,Sydney - City and Inner South,Males,15-24 years,Employed total,11.087451
4,2012-01-01,118,Sydney - Eastern Suburbs,Males,15-24 years,Employed total,8.474928


## 5. Save three cleaned source tables

Each source workbook gets its own cleaned CSV named for the ABS table and reference month, for example `table_010_2026-08.csv`. The release month is recorded in the filename and in `sources.csv`, not as a column.

| File | Columns |
|---|---|
| `table_010_*.csv`, `table_014_*.csv` | `date`, `series_id`, `measure`, `sex`, `age`, `region`, `series_type`, `data_type`, `unit`, `value` |
| `mlf1_*.csv` | `date`, `measure`, `sex`, `age`, `sa4_code`, `sa4_region`, `series_type`, `unit`, `value` |

`region` in Tables 010 and 014 is Australia or a state/territory. `sources.csv` records each ABS download and its output file; it is written last, so it only exists for completed runs.

In [7]:
cols = ['date', 'series_id', 'measure', 'sex', 'age', 'region', 'series_type', 'data_type', 'unit', 'value']
exports = {d: f[cols] for d, f in ts.groupby('dataset')}
exports['MLF1'] = (
    mlf[['date', 'measure', 'sex', 'age', 'region_code', 'region', 'series_type', 'unit', 'value']]
       .rename(columns={'region_code': 'sa4_code', 'region': 'sa4_region'})
)

for dataset, frame in exports.items():
    sources.loc[dataset, 'output_file'] = f"{dataset.lower().replace(' ', '_')}_{reference_period:%Y-%m}.csv"
    frame.to_csv(run / sources.loc[dataset, 'output_file'], index=False, date_format='%Y-%m-%d')

sources.to_csv(run / 'sources.csv')  # Written last: its presence marks the run complete.
print(f'Saved {len(exports)} cleaned CSV files for {reference_period:%B %Y} in {run}')
display(sources[['download_file', 'output_file']])

with pd.option_context('display.max_columns', None):
    for dataset, frame in exports.items():
        print(f"\n{sources.loc[dataset, 'output_file']} — {frame.shape[0]:,} rows × {frame.shape[1]} columns")
        display(frame.head())

Saved 3 cleaned CSV files for August 2026 in /Users/aristotle/Documents/ChatGPT/LISA Back End/data-analysis-best-practices/.claude/worktrees/abs-labour-force-automation-b9a616/outputs/abs_labour_force/20260926T101455631009Z


,download_file,output_file
dataset,,
Table 010,62020010.xlsx,table_010_2026-08.csv
Table 014,62020014.xlsx,table_014_2026-08.csv
MLF1,MLF1.xlsx,mlf1_2026-08.csv



table_010_2026-08.csv — 433,752 rows × 10 columns


,date,series_id,measure,sex,age,region,series_type,data_type,unit,value
0,1978-02-01,A84423127L,Employed total,Persons,15 years and over,Australia,Trend,STOCK,000,6007.747586
1,1978-03-01,A84423127L,Employed total,Persons,15 years and over,Australia,Trend,STOCK,000,6014.722083
2,1978-04-01,A84423127L,Employed total,Persons,15 years and over,Australia,Trend,STOCK,000,6021.174221
3,1978-05-01,A84423127L,Employed total,Persons,15 years and over,Australia,Trend,STOCK,000,6026.885932
4,1978-06-01,A84423127L,Employed total,Persons,15 years and over,Australia,Trend,STOCK,000,6030.961427



table_014_2026-08.csv — 43,650 rows × 10 columns


,date,series_id,measure,sex,age,region,series_type,data_type,unit,value
433752,1986-04-01,A84424617W,Employed total,Persons,15-24 years,Australia,Original,STOCK,000,1652.431944
433753,1986-05-01,A84424617W,Employed total,Persons,15-24 years,Australia,Original,STOCK,000,1642.290159
433754,1986-06-01,A84424617W,Employed total,Persons,15-24 years,Australia,Original,STOCK,000,1629.630142
433755,1986-07-01,A84424617W,Employed total,Persons,15-24 years,Australia,Original,STOCK,000,1610.522466
433756,1986-08-01,A84424617W,Employed total,Persons,15-24 years,Australia,Original,STOCK,000,1583.597332



mlf1_2026-08.csv — 551,232 rows × 9 columns


,date,measure,sex,age,sa4_code,sa4_region,series_type,unit,value
0,2012-01-01,Employed total,Males,15-24 years,102,Central Coast,Modelled,000,12.757092
1,2012-01-01,Employed total,Males,15-24 years,115,Sydney - Baulkham Hills and Hawkesbury,Modelled,000,9.284133
2,2012-01-01,Employed total,Males,15-24 years,116,Sydney - Blacktown,Modelled,000,12.403918
3,2012-01-01,Employed total,Males,15-24 years,117,Sydney - City and Inner South,Modelled,000,11.087451
4,2012-01-01,Employed total,Males,15-24 years,118,Sydney - Eastern Suburbs,Modelled,000,8.474928
